In [4]:
import pandas as pd
import json
import time
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. Setup API
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENAI")
client = OpenAI(api_key=api_key)

# 2. Load file
file_path = '/kaggle/input/datasets/ianaaa/empty-dataset-200/annotation_pilot_coderA.csv'
df = pd.read_csv(file_path)

# Convert columns to 'object' type
annotation_cols = [
    'include_pair', 'speaker_perspective', 'target_boomer', 'target_genx', 
    'target_millennial', 'target_genz', 'parent_genericity', 'parent_valence', 
    'Parent text Frame', 'reply_stance', 'reply_counter_reframing', 'notes'
]
for col in annotation_cols:
    if col in df.columns:
        df[col] = df[col].astype(object)

# 3. Prompt mapping JSON keys to CSV columns
SYSTEM_PROMPT = """
You are an expert sociologist. Analyze the Reddit thread for generational stereotypes.
Return your analysis as a JSON object where the keys match the specific CSV column names provided below.

COLUMN VALUES & MAPPING RULES:
- include_pair: 'YES' if it contains a generational claim, 'NO' otherwise.
- speaker_perspective: 'Auto', 'Hetero', or 'Mixed'.
- target_boomer, target_genx, target_millennial, target_genz: use 1 if targeted, 0 if not.
- parent_genericity: 'Generic' or 'Specific'.
- parent_valence: 'Positive', 'Negative', 'Mixed', or 'Neutral'.
- Parent text Frame: Use one: Work/Economy, Wealth/Housing, Tech/Media, Politics, Education, Health/Aging, Family, Environment, Identity.
- reply_stance: 'Support', 'Mitigate', 'Oppose', or 'Neutral / Off-topic'.
- reply_counter_reframing: 1 if reply uses reframing (appealing to morals/empathy), 0 otherwise.
- notes: Briefly explain your reasoning (diagnostic causes/moral evaluation).

OUTPUT FORMAT: JSON only.
"""

def annotate_and_map(row):
    user_content = f"Title: {row['post_title']}\nText: {row['post_selftext']}\nParent: {row['parent_text']}\nReply: {row['reply_text']}"
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o", 
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            response_format={ "type": "json_object" }
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error at index {row.name}: {e}")
        return None

# 4. Processing Loop
print(f"Annotating {len(df)} rows into existing columns...")

for index, row in df.iterrows():
    result = annotate_and_map(row)
    if result:
        for key in result:
            if key in df.columns:
                df.at[index, key] = result[key]
    
    if index % 10 == 0:
        print(f"Completed {index}/200...")
    time.sleep(0.5)

# 5. Save the populated file
df.to_csv('final_populated_pilot.csv', index=False)
print("Finished! File saved as 'final_populated_pilot.csv' with all original columns filled.")

Annotating 200 rows into existing columns...
Completed 0/200...
Completed 10/200...
Completed 20/200...
Completed 30/200...
Completed 40/200...
Completed 50/200...
Completed 60/200...
Completed 70/200...
Completed 80/200...
Completed 90/200...
Completed 100/200...
Completed 110/200...
Completed 120/200...
Completed 130/200...
Completed 140/200...
Completed 150/200...
Completed 160/200...
Completed 170/200...
Completed 180/200...
Completed 190/200...
Finished! File saved as 'final_populated_pilot.csv' with all original columns filled.


In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

# 1. LOAD THE DATASETS
df_gpt = pd.read_csv('/kaggle/working/final_populated_pilot.csv')
df_gold = pd.read_csv('/kaggle/input/golden-standard/golden_standard.csv', sep=';')

# Clean up column names (remove leading/trailing spaces)
df_gold.columns = [c.strip() for c in df_gold.columns]

# 2. DEFINE NORMALIZATION FUNCTIONS
def norm_val(val):
    val = str(val).lower().strip()
    if val in ['neg', 'negative']: return 'negative'
    if val in ['pos', 'positive']: return 'positive'
    if val in ['mixed']: return 'mixed'
    if val in ['neutral', '0', '0.0']: return 'neutral'
    return 'neutral'

def norm_include(val):
    val = str(val).lower().strip()
    return 'yes' if 'yes' in val else 'no'

def norm_frame(val):
    return str(val).lower().strip().replace('_', '/').replace(' ', '')

def norm_stance(val):
    val = str(val).lower().strip()
    if 'neutral' in val or 'off-topic' in val: return 'neutral'
    return val

# 3. PREPARE THE DATA FOR COMPARISON
comparison_df = pd.DataFrame()

# Column Mapping: [GPT Column Name] -> [Gold Standard Column Name]
mapping = {
    'include_pair': 'Include? (yes/no)',
    'speaker_perspective': 'Perspective (auto/hetero/mixed)',
    'parent_valence': 'Parent: Valence (neg/pos/mixed/neutral)',
    'reply_stance': 'Reply: Stance (support/mitigate/oppose/neutral)',
    'Parent text Frame': 'Parent: Frame (pick one)'
}

# Apply normalization
comparison_df['include_gpt'] = df_gpt['include_pair'].apply(norm_include)
comparison_df['include_gold'] = df_gold['Include? (yes/no)'].apply(norm_include)

comparison_df['valence_gpt'] = df_gpt['parent_valence'].apply(norm_val)
comparison_df['valence_gold'] = df_gold['Parent: Valence (neg/pos/mixed/neutral)'].apply(norm_val)

comparison_df['frame_gpt'] = df_gpt['Parent text Frame'].apply(norm_frame)
comparison_df['frame_gold'] = df_gold['Parent: Frame (pick one)'].apply(norm_frame)

comparison_df['stance_gpt'] = df_gpt['reply_stance'].apply(norm_stance)
comparison_df['stance_gold'] = df_gold['Reply: Stance (support/mitigate/oppose/neutral)'].apply(norm_stance)

# 4. CALCULATE KAPPA
print("--- OVERALL DETECTION AGREEMENT (N=200) ---")
k_detect = cohen_kappa_score(comparison_df['include_gpt'], comparison_df['include_gold'])
print(f"Include/Exclude Kappa: {k_detect:.4f}")

print("\n--- LABEL AGREEMENT (Only where both agreed to INCLUDE, N=~80) ---")
common_yes = (comparison_df['include_gpt'] == 'yes') & (comparison_df['include_gold'] == 'yes')
df_yes = comparison_df[common_yes]

for category in ['valence', 'frame', 'stance']:
    k = cohen_kappa_score(df_yes[f'{category}_gpt'], df_yes[f'{category}_gold'])
    print(f"{category.capitalize()} Kappa: {k:.4f}")

# 5. ERROR ANALYSIS
disagreements = df_yes[df_yes['frame_gpt'] != df_yes['frame_gold']]
print(f"\nNumber of Frame Disagreements: {len(disagreements)}")

--- OVERALL DETECTION AGREEMENT (N=200) ---
Include/Exclude Kappa: 0.4763

--- LABEL AGREEMENT (Only where both agreed to INCLUDE, N=~80) ---
Valence Kappa: 0.4623
Frame Kappa: 0.2449
Stance Kappa: 0.3930

Number of Frame Disagreements: 57
